In [1]:
"""
This train/eval notebook contains the new iteration of the
CORAL experiment in which source data is adapted to resemble
target (source to target) and classifiers fitted on both the
original and adapted source data are evaluated on the target.
"""
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, recall_score
import joblib
import os

In [2]:
# Load datasets and pretrained artifacts
source_train_data_path = os.path.join('data', 'processed', 'source', 'source_train.csv')
source_test_data_path = os.path.join('data', 'processed', 'source', 'source_test.csv')
target_train_data_path = os.path.join('data', 'processed', 'target', 'target_train.csv')
target_test_data_path = os.path.join('data', 'processed', 'target', 'target_test.csv')

label_encoder_path = os.path.join('models', 'label_encoder.joblib')
coral_source_stats_path = os.path.join('models', 'global_coral_source_stats.joblib')
coral_target_stats_path = os.path.join('models', 'global_coral_target_stats.joblib')

source_train_df = pd.read_csv(source_train_data_path)
source_test_df = pd.read_csv(source_test_data_path)
target_train_df = pd.read_csv(target_train_data_path)
target_test_df = pd.read_csv(target_test_data_path)

label_encoder = joblib.load(label_encoder_path)
coral_source_stats = joblib.load(coral_source_stats_path)
coral_target_stats = joblib.load(coral_target_stats_path)

print(f"Source train shape: {source_train_df.shape}")
print(f"Source test shape: {source_test_df.shape}")
print(f"Target train shape: {target_train_df.shape}")
print(f"Target test shape: {target_test_df.shape}")
print("Loaded artifacts: label_encoder, global_coral_source_stats, global_coral_target_stats")

Source train shape: (1259420, 71)
Source test shape: (314856, 71)
Target train shape: (144714, 71)
Target test shape: (36179, 71)
Loaded artifacts: label_encoder, global_coral_source_stats, global_coral_target_stats


In [3]:
# Sanity checks
# Verify source and target datasets have equivalent feature and label space.

shared_feature_path = os.path.join('data', 'processed', 'shared_feature_space.json')
shared_label_path = os.path.join('data', 'processed', 'shared_label_space.json')

import json
with open(shared_feature_path, 'r') as f:
    shared_feature_payload = json.load(f)
with open(shared_label_path, 'r') as f:
    shared_label_payload = json.load(f)

# Accept either list format or object payload for schema compatibility with preprocessing notebooks.
if isinstance(shared_feature_payload, dict):
    if 'features' not in shared_feature_payload:
        raise ValueError("Expected key 'features' when shared feature payload is a JSON object.")
    shared_features = list(shared_feature_payload['features'])
elif isinstance(shared_feature_payload, list):
    shared_features = list(shared_feature_payload)
else:
    raise ValueError(f"Unsupported shared feature payload type: {type(shared_feature_payload).__name__}")

if isinstance(shared_label_payload, dict):
    if 'labels' not in shared_label_payload:
        raise ValueError("Expected key 'labels' when shared label payload is a JSON object.")
    shared_labels = list(shared_label_payload['labels'])
elif isinstance(shared_label_payload, list):
    shared_labels = list(shared_label_payload)
else:
    raise ValueError(f"Unsupported shared label payload type: {type(shared_label_payload).__name__}")

assert set(shared_features) <= set(source_train_df.columns), "Source train missing shared features!"
assert set(shared_features) <= set(source_test_df.columns), "Source test missing shared features!"
assert set(shared_features) <= set(target_train_df.columns), "Target train missing shared features!"
assert set(shared_features) <= set(target_test_df.columns), "Target test missing shared features!"

# Normalize each split's labels into class-name space before set comparison.
def to_label_name_set(label_series, fitted_label_encoder):
    labels = label_series.dropna()
    if pd.api.types.is_numeric_dtype(labels):
        return set(fitted_label_encoder.inverse_transform(labels.astype(int).to_numpy()))
    return set(labels.astype(str).to_numpy())

expected_label_set = set(map(str, shared_labels))
source_train_label_set = to_label_name_set(source_train_df['Label'], label_encoder)
source_test_label_set = to_label_name_set(source_test_df['Label'], label_encoder)
target_train_label_set = to_label_name_set(target_train_df['Label'], label_encoder)
target_test_label_set = to_label_name_set(target_test_df['Label'], label_encoder)

source_combined_label_set = source_train_label_set | source_test_label_set
target_combined_label_set = target_train_label_set | target_test_label_set

assert source_combined_label_set == expected_label_set, (
    "Combined source label space mismatch vs shared labels! "
    f"Missing={sorted(expected_label_set - source_combined_label_set)}, "
    f"Extra={sorted(source_combined_label_set - expected_label_set)}"
 )
assert target_combined_label_set == expected_label_set, (
    "Combined target label space mismatch vs shared labels! "
    f"Missing={sorted(expected_label_set - target_combined_label_set)}, "
    f"Extra={sorted(target_combined_label_set - expected_label_set)}"
 )
assert source_combined_label_set == target_combined_label_set, (
    "Combined source/target label spaces do not match!"
 )

print("Sanity checks passed: Shared feature space verified for source train/test and target train/test.")
print("Sanity checks passed: Combined source and target label spaces match shared labels.")

Sanity checks passed: Shared feature space verified for source train/test and target train/test.
Sanity checks passed: Combined source and target label spaces match shared labels.


In [14]:
### Calculate (*GLOBAL) CORAL transform and apply to both source splits (train and test) ###
# Note: if class-wise coral is needed, comment out this section (universal variable
# names are used in both coral cells and used downstream in evaluation cells)

def stable_symmetric_matrix_power(matrix, power, eps=1e-6):
    """Return a numerically stabilized symmetric matrix power."""
    matrix = np.asarray(matrix, dtype=np.float64)
    matrix = (matrix + matrix.T) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(matrix)
    ridge = max(eps, float(-eigenvalues.min() + eps)) if eigenvalues.min() <= 0 else eps
    clipped_eigenvalues = np.clip(eigenvalues + ridge, eps, None)

    powered = eigenvectors @ np.diag(clipped_eigenvalues ** power) @ eigenvectors.T
    return (powered + powered.T) / 2.0, float(eigenvalues.min()), ridge


# Validate stats and feature ordering before applying CORAL.
source_feature_order = coral_source_stats.get("feature_order")
target_feature_order = coral_target_stats.get("feature_order")

if source_feature_order is None or target_feature_order is None:
    raise KeyError("CORAL stats must include 'feature_order' metadata.")
if list(source_feature_order) != list(target_feature_order):
    raise ValueError("Source and target CORAL stats feature_order do not match.")
if list(shared_features) != list(source_feature_order):
    raise ValueError(
        "Runtime shared feature order does not match CORAL stats feature_order. "
        "Regenerate stats or align feature ordering before evaluation."
    )

# Extract source/target statistics used by the source-to-target CORAL transform.
source_mean = np.asarray(coral_source_stats["mean"], dtype=np.float64)
source_cov = np.asarray(coral_source_stats["covariance"], dtype=np.float64)

target_mean = np.asarray(coral_target_stats["mean"], dtype=np.float64)
target_cov = np.asarray(coral_target_stats["covariance"], dtype=np.float64)

n_features = len(source_feature_order)
if source_mean.shape[0] != n_features or target_mean.shape[0] != n_features:
    raise ValueError("CORAL mean vector length does not match feature_order length.")
if source_cov.shape != (n_features, n_features) or target_cov.shape != (n_features, n_features):
    raise ValueError("CORAL covariance shape does not match feature_order length.")

# Convert source features to numpy using the validated canonical feature order.
X_source_train_np = source_train_df[source_feature_order].to_numpy(dtype=np.float64)
X_source_test_np = source_test_df[source_feature_order].to_numpy(dtype=np.float64)

# Center source data using source mean before CORAL transform.
X_source_train_centered = X_source_train_np - source_mean
X_source_test_centered = X_source_test_np - source_mean

# Build the source-to-target CORAL transform: A = Cs^(-1/2) * Ct^(1/2).
source_cov_inv_sqrt, source_min_eig, source_ridge = stable_symmetric_matrix_power(source_cov, -0.5)
target_cov_sqrt, target_min_eig, target_ridge = stable_symmetric_matrix_power(target_cov, 0.5)

A_coral = source_cov_inv_sqrt @ target_cov_sqrt

# Apply the same transform to both source train and source test splits.
X_source_train_coral_np = (X_source_train_centered @ A_coral) + target_mean
X_source_test_coral_np = (X_source_test_centered @ A_coral) + target_mean

if not np.isfinite(X_source_train_coral_np).all():
    raise ValueError("CORAL transform produced non-finite values for source train split.")
if not np.isfinite(X_source_test_coral_np).all():
    raise ValueError("CORAL transform produced non-finite values for source test split.")

X_source_train_coral_df = pd.DataFrame(
    X_source_train_coral_np,
    columns=source_feature_order,
    index=source_train_df.index,
)
X_source_test_coral_df = pd.DataFrame(
    X_source_test_coral_np,
    columns=source_feature_order,
    index=source_test_df.index,
)

print("CORAL source-to-target transform computed successfully.")
print(f"Feature count: {n_features}")
print(f"Source covariance min eig / ridge: {source_min_eig:.6e} / {source_ridge:.6e}")
print(f"Target covariance min eig / ridge: {target_min_eig:.6e} / {target_ridge:.6e}")
print(f"Adapted source train shape: {X_source_train_coral_df.shape}")
print(f"Adapted source test shape: {X_source_test_coral_df.shape}")

CORAL source-to-target transform computed successfully.
Feature count: 70
Source covariance min eig / ridge: 1.000665e-02 / 1.000000e-06
Target covariance min eig / ridge: 1.145981e-02 / 1.000000e-06
Adapted source train shape: (1259420, 70)
Adapted source test shape: (314856, 70)


In [15]:
# ### Calculate (*PER-CLASS) CORAL transform and apply to both source splits (train and test) ###

# # Note: if global coral is needed, comment out this section (universal variable
# # names are used in both coral cells and used downstream in evaluation cells)

# def stable_symmetric_matrix_power(matrix, power, eps=1e-6):
#     """Return a numerically stabilized symmetric matrix power."""
#     matrix = np.asarray(matrix, dtype=np.float64)
#     matrix = (matrix + matrix.T) / 2.0

#     eigenvalues, eigenvectors = np.linalg.eigh(matrix)
#     ridge = max(eps, float(-eigenvalues.min() + eps)) if eigenvalues.min() <= 0 else eps
#     clipped_eigenvalues = np.clip(eigenvalues + ridge, eps, None)

#     powered = eigenvectors @ np.diag(clipped_eigenvalues ** power) @ eigenvectors.T
#     return (powered + powered.T) / 2.0, float(eigenvalues.min()), ridge

# # Load per-class CORAL statistics from preprocessing artifacts.
# perclass_source_stats_path = os.path.join('models', 'perclass_coral_source_stats.joblib')
# perclass_target_stats_path = os.path.join('models', 'perclass_coral_target_stats.joblib')

# if not os.path.exists(perclass_source_stats_path):
#     raise FileNotFoundError(f"Per-class source CORAL stats not found at {perclass_source_stats_path}")
# if not os.path.exists(perclass_target_stats_path):
#     raise FileNotFoundError(f"Per-class target CORAL stats not found at {perclass_target_stats_path}")

# perclass_source_stats = joblib.load(perclass_source_stats_path)
# perclass_target_stats = joblib.load(perclass_target_stats_path)

# def get_canonical_feature_order(perclass_stats, stats_name):
#     """
#     Extract the canonical feature ordering from a per-class CORAL stats artifact.

#     This is used during evaluation to verify that the source and target per-class
#     statistics were generated from the same feature schema before applying any
#     class-conditional CORAL transform to the source splits.
#     """
#     if not isinstance(perclass_stats, dict) or not perclass_stats:
#         raise ValueError(f"{stats_name} must be a non-empty dict of per-class statistics.")

#     feature_orders = {}
#     for class_name, stats in perclass_stats.items():
#         feature_order = stats.get('feature_order')
#         if feature_order is None:
#             raise KeyError(f"{stats_name}[{class_name!r}] is missing 'feature_order'.")
#         feature_orders[class_name] = tuple(feature_order)

#     unique_orders = set(feature_orders.values())
#     if len(unique_orders) != 1:
#         raise ValueError(f"{stats_name} contains inconsistent feature_order values across classes.")

#     return list(next(iter(unique_orders)))

# source_feature_order = get_canonical_feature_order(perclass_source_stats, 'perclass_source_stats')
# target_feature_order = get_canonical_feature_order(perclass_target_stats, 'perclass_target_stats')

# if source_feature_order != target_feature_order:
#     raise ValueError("Source and target per-class CORAL feature_order values do not match.")
# if list(shared_features) != source_feature_order:
#     raise ValueError("Runtime shared feature order does not match per-class CORAL stats feature_order.")

# # Preallocate adapted source DataFrames so row order and feature order stay fixed.
# X_source_train_coral_df = pd.DataFrame(
#     index=source_train_df.index,
#     columns=source_feature_order,
#     dtype=np.float64,
# )
# X_source_test_coral_df = pd.DataFrame(
#     index=source_test_df.index,
#     columns=source_feature_order,
#     dtype=np.float64,
# )

# X_source_train_np = source_train_df[source_feature_order].to_numpy(dtype=np.float64)
# X_source_test_np = source_test_df[source_feature_order].to_numpy(dtype=np.float64)

# y_source_train_int = source_train_df['Label'].astype(int).to_numpy()
# y_source_test_int = source_test_df['Label'].astype(int).to_numpy()

# class_summaries = []
# missing_classes = []

# for class_id in sorted(np.unique(y_source_train_int)):
#     class_name = str(label_encoder.inverse_transform([int(class_id)])[0])
    
#     source_stats = perclass_source_stats.get(class_name)
#     target_stats = perclass_target_stats.get(class_name)
#     if source_stats is None or target_stats is None:
#         missing_classes.append(class_name)
#         continue

#     source_mean = np.asarray(source_stats['mean'], dtype=np.float64)
#     source_cov = np.asarray(source_stats['covariance'], dtype=np.float64)
#     target_mean = np.asarray(target_stats['mean'], dtype=np.float64)
#     target_cov = np.asarray(target_stats['covariance'], dtype=np.float64)

#     if source_mean.shape[0] != len(source_feature_order) or target_mean.shape[0] != len(source_feature_order):
#         raise ValueError(f"Per-class CORAL mean length mismatch for class {class_name}.")
#     if source_cov.shape != (len(source_feature_order), len(source_feature_order)) or target_cov.shape != (len(source_feature_order), len(source_feature_order)):
#         raise ValueError(f"Per-class CORAL covariance shape mismatch for class {class_name}.")

#     source_cov_inv_sqrt, source_min_eig, source_ridge = stable_symmetric_matrix_power(source_cov, -0.5)
#     target_cov_sqrt, target_min_eig, target_ridge = stable_symmetric_matrix_power(target_cov, 0.5)
#     A_coral = source_cov_inv_sqrt @ target_cov_sqrt

#     train_class_mask = y_source_train_int == class_id
#     train_class_count = int(train_class_mask.sum())
#     if train_class_count > 0:
#         X_source_train_class = X_source_train_np[train_class_mask, :]
#         X_source_train_class_centered = X_source_train_class - source_mean
#         X_source_train_class_coral = (X_source_train_class_centered @ A_coral) + target_mean
        
#         if not np.isfinite(X_source_train_class_coral).all():
#             raise ValueError(f"Per-class CORAL transform produced non-finite values for class {class_name} in source train split.")
        
#         X_source_train_coral_df.loc[train_class_mask, source_feature_order] = X_source_train_class_coral

#     test_class_mask = y_source_test_int == class_id
#     test_class_count = int(test_class_mask.sum())
#     if test_class_count > 0:
#         X_source_test_class = X_source_test_np[test_class_mask, :]
#         X_source_test_class_centered = X_source_test_class - source_mean
#         X_source_test_class_coral = (X_source_test_class_centered @ A_coral) + target_mean
        
#         if not np.isfinite(X_source_test_class_coral).all():
#             raise ValueError(f"Per-class CORAL transform produced non-finite values for class {class_name} in source test split.")
        
#         X_source_test_coral_df.loc[test_class_mask, source_feature_order] = X_source_test_class_coral

#     class_summaries.append({
#         'class_id': int(class_id),
#         'class_name': class_name,
#         'train_sample_count': train_class_count,
#         'test_sample_count': test_class_count,
#         'source_min_eig': source_min_eig,
#         'target_min_eig': target_min_eig,
#         'source_ridge': source_ridge,
#         'target_ridge': target_ridge,
#         'source_shrinkage': float(source_stats.get('covariance_shrinkage', np.nan)),
#         'target_shrinkage': float(target_stats.get('covariance_shrinkage', np.nan)),
#     })

# if missing_classes:
#     raise ValueError(
#         'Per-class CORAL stats are missing for these classes: '
#         f"{sorted(set(missing_classes))}. Check the per-class preprocessing artifacts."
#     )

# if X_source_train_coral_df.isna().any().any() or X_source_test_coral_df.isna().any().any():
#     raise ValueError("Per-class CORAL adaptation left NaN values in the adapted source matrices.")

# X_source_train_coral_df = X_source_train_coral_df.astype(np.float64)
# X_source_test_coral_df = X_source_test_coral_df.astype(np.float64)

# print("Per-class CORAL source-to-target transform computed successfully.")
# print(f"Feature count: {len(source_feature_order)}")
# print(f"Classes adapted: {len(class_summaries)}")
# for summary in class_summaries:
#     print(
#         f"- {summary['class_name']}: "
#         f"train_n={summary['train_sample_count']}, test_n={summary['test_sample_count']}, "
#         f"source_shrinkage={summary['source_shrinkage']:.6f}, "
#         f"target_shrinkage={summary['target_shrinkage']:.6f}"
#     )
# print(f"Adapted source train shape: {X_source_train_coral_df.shape}")
# print(f"Adapted source test shape: {X_source_test_coral_df.shape}")

In [16]:
### Extract CORAL-aligned source data ###
# Note: to be used for domain shift empirical measurement. Export the active
# CORAL-adapted source splits to data/processed/source.

# This cell is intentionally compatible with the comment-out variable naming
# strategy for the two CORAL variants implemented in the preceding cells:
# whichever CORAL variant cell is active will populate these variables and get
# exported here.

export_dir = os.path.join('data', 'processed', 'source')
os.makedirs(export_dir, exist_ok=True)

if 'X_source_train_coral_df' not in globals() or 'X_source_test_coral_df' not in globals():
    raise ValueError(
        "Missing X_source_train_coral_df / X_source_test_coral_df in memory. "
        "Run exactly one CORAL transform cell (global or per-class) before export."
    )

if not isinstance(X_source_train_coral_df, pd.DataFrame) or not isinstance(X_source_test_coral_df, pd.DataFrame):
    raise TypeError("X_source_train_coral_df and X_source_test_coral_df must both be pandas DataFrames.")

if X_source_train_coral_df.shape[0] != len(source_train_df) or X_source_test_coral_df.shape[0] != len(source_test_df):
    raise ValueError(
        "CORAL DataFrame row count mismatch with source splits. "
        f"Expected train/test rows {len(source_train_df)}/{len(source_test_df)}, "
        f"got {X_source_train_coral_df.shape[0]}/{X_source_test_coral_df.shape[0]}."
    )

if X_source_train_coral_df.isna().any().any() or X_source_test_coral_df.isna().any().any():
    raise ValueError("Active CORAL-adapted source DataFrames contain NaN values.")

coral_train_export_df = X_source_train_coral_df.copy()
coral_test_export_df = X_source_test_coral_df.copy()
coral_train_export_df['Label'] = source_train_df['Label'].astype(int).to_numpy()
coral_test_export_df['Label'] = source_test_df['Label'].astype(int).to_numpy()

# ***Note: modify export name according to active CORAL variant (per-class or global).
coral_train_export_path = os.path.join(export_dir, 'global-coral-source-train.csv')
coral_test_export_path = os.path.join(export_dir, 'global-coral-source-test.csv')

coral_train_export_df.to_csv(coral_train_export_path, index=False)
coral_test_export_df.to_csv(coral_test_export_path, index=False)

print('Exported active CORAL-aligned source splits:')
print(f"- Train: {coral_train_export_path} | shape={X_source_train_coral_df.shape}")
print(f"- Test: {coral_test_export_path} | shape={X_source_test_coral_df.shape}")

Exported active CORAL-aligned source splits:
- Train: data/processed/source/global-coral-source-train.csv | shape=(1259420, 70)
- Test: data/processed/source/global-coral-source-test.csv | shape=(314856, 70)


In [17]:
### Fit classifiers on original (no-CORAL) source train split ###


### Random Forest ###
# hyperparameters: n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1
from sklearn.ensemble import RandomForestClassifier

X_source_train = source_train_df[shared_features]
y_source_train = source_train_df['Label'].astype(int)

rf_no_coral = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
 )
rf_no_coral.fit(X_source_train, y_source_train)

rf_no_coral_model_path = os.path.join('models', 'rf_no_coral_model.joblib')
joblib.dump(rf_no_coral, rf_no_coral_model_path)

print('Random Forest trained on source train split.')
print(f'Saved Random Forest model to: {rf_no_coral_model_path}')

### SVM ###

# hyperparameters: kernel=linear, C=1.5
X_source_train = source_train_df[shared_features]
y_source_train = source_train_df['Label'].astype(int)

svm_no_coral = LinearSVC(
    C=1.5,
    tol=1e-2,
    dual=False,
    max_iter=1000, # reduced from 2000 6/16/26
    random_state=42,
 )
svm_no_coral.fit(X_source_train, y_source_train)

svm_no_coral_model_path = os.path.join('models', 'svm_no_coral_model.joblib')
joblib.dump(svm_no_coral, svm_no_coral_model_path)

print('Linear SVM trained on source train split.')
print(f'Saved Linear SVM model to: {svm_no_coral_model_path}')

### MLP ###

# hyperparameters: hidden_layers_sizes=(64,), alpha=0.001
from sklearn.neural_network import MLPClassifier

X_source_train = source_train_df[shared_features]
y_source_train = source_train_df['Label'].astype(int)

mlp_no_coral = MLPClassifier(
    hidden_layer_sizes=(64,),
    alpha=0.001,
    random_state=42,
    max_iter=200,
 )
mlp_no_coral.fit(X_source_train, y_source_train)

mlp_no_coral_model_path = os.path.join('models', 'mlp_no_coral_model.joblib')
joblib.dump(mlp_no_coral, mlp_no_coral_model_path)

print('MLP trained on source train split.')
print(f'Saved MLP model to: {mlp_no_coral_model_path}')

Random Forest trained on source train split.
Saved Random Forest model to: models/rf_no_coral_model.joblib


/Users/amazlumyan/Desktop/GitHub/domain-adaptation-poc/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Linear SVM trained on source train split.
Saved Linear SVM model to: models/svm_no_coral_model.joblib
MLP trained on source train split.
Saved MLP model to: models/mlp_no_coral_model.joblib


In [18]:
### Fit classifiers on adapted (with-CORAL) source train split ###

### Random Forest ###
# hyperparameters: n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1
from sklearn.ensemble import RandomForestClassifier

X_source_train_coral = X_source_train_coral_df
y_source_train_coral = source_train_df['Label'].astype(int)

rf_coral = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
 )
rf_coral.fit(X_source_train_coral, y_source_train_coral)

rf_coral_model_path = os.path.join('models', 'rf_coral_model.joblib')
joblib.dump(rf_coral, rf_coral_model_path)

print('Random Forest trained on source train split.')
print(f'Saved Random Forest model to: {rf_coral_model_path}')

### SVM ###

# hyperparameters: kernel=linear, C=1.5
X_source_train_coral = X_source_train_coral_df
y_source_train_coral = source_train_df['Label'].astype(int)

svm_coral = LinearSVC(
    C=1.5,
    tol=1e-2,
    dual=False,
    max_iter=1000,
    random_state=42,
 )
svm_coral.fit(X_source_train_coral, y_source_train_coral)

svm_coral_model_path = os.path.join('models', 'svm_coral_model.joblib')
joblib.dump(svm_coral, svm_coral_model_path)

print('Linear SVM trained on source train split.')
print(f'Saved Linear SVM model to: {svm_coral_model_path}')

### MLP ###

# hyperparameters: hidden_layers_sizes=(64,), alpha=0.001
from sklearn.neural_network import MLPClassifier

X_source_train_coral = X_source_train_coral_df
y_source_train_coral = source_train_df['Label'].astype(int)

mlp_coral = MLPClassifier(
    hidden_layer_sizes=(64,),
    alpha=0.001,
    random_state=42,
    max_iter=200,
 )
mlp_coral.fit(X_source_train_coral, y_source_train_coral)

mlp_coral_model_path = os.path.join('models', 'mlp_coral_model.joblib')
joblib.dump(mlp_coral, mlp_coral_model_path)

print('MLP trained on source train split.')
print(f'Saved MLP model to: {mlp_coral_model_path}')

Random Forest trained on source train split.
Saved Random Forest model to: models/rf_coral_model.joblib
Linear SVM trained on source train split.
Saved Linear SVM model to: models/svm_coral_model.joblib
MLP trained on source train split.
Saved MLP model to: models/mlp_coral_model.joblib


In [19]:
### Evaluate no-CORAL classifiers on no-CORAL source ###

# Load no-CORAL models if not already in memory.
rf_no_coral_model_path = os.path.join('models', 'rf_no_coral_model.joblib')
svm_no_coral_model_path = os.path.join('models', 'svm_no_coral_model.joblib')
mlp_no_coral_model_path = os.path.join('models', 'mlp_no_coral_model.joblib')

loaded_models = []
if 'rf_no_coral' not in globals() or rf_no_coral is None:
    rf_no_coral = joblib.load(rf_no_coral_model_path)
    loaded_models.append('rf_no_coral')
if 'svm_no_coral' not in globals() or svm_no_coral is None:
    svm_no_coral = joblib.load(svm_no_coral_model_path)
    loaded_models.append('svm_no_coral')
if 'mlp_no_coral' not in globals() or mlp_no_coral is None:
    mlp_no_coral = joblib.load(mlp_no_coral_model_path)
    loaded_models.append('mlp_no_coral')

if loaded_models:
    print(f"Loaded no-CORAL models from disk: {', '.join(loaded_models)}")
else:
    print('No-CORAL models already available in memory.')

X_source_test = source_test_df[shared_features]
y_source_test = source_test_df['Label'].astype(int)
X_target_test = target_test_df[shared_features]
y_target_test = target_test_df['Label'].astype(int)

print('==============================================')
print('NO-CORAL CLASSIFIERS ON NO-CORAL SOURCE')
print('==============================================')

# Evaluate the no-coral RF on the unaltered source test split.
y_pred_source = rf_no_coral.predict(X_source_test)
source_accuracy = accuracy_score(y_source_test, y_pred_source)
source_macro_recall = recall_score(y_source_test, y_pred_source, average='macro', zero_division=0)
print('[S-trained RF on S_test]')
print(f'Accuracy: {source_accuracy:.6f}')
print(f'Macro Recall: {source_macro_recall:.6f}')
print('Classification Report:')
print(classification_report(y_source_test, y_pred_source, target_names=label_encoder.classes_, zero_division=0))

# Evaluate the no-coral SVM on the unaltered source test split.
y_pred_source = svm_no_coral.predict(X_source_test)
source_accuracy = accuracy_score(y_source_test, y_pred_source)
source_macro_recall = recall_score(y_source_test, y_pred_source, average='macro', zero_division=0)
print('[S-trained SVM on S_test]')
print(f'Accuracy: {source_accuracy:.6f}')
print(f'Macro Recall: {source_macro_recall:.6f}')
print('Classification Report:')
print(classification_report(y_source_test, y_pred_source, target_names=label_encoder.classes_, zero_division=0))

# Evaluate the no-coral MLP on the unaltered source test split.
y_pred_source = mlp_no_coral.predict(X_source_test)
source_accuracy = accuracy_score(y_source_test, y_pred_source)
source_macro_recall = recall_score(y_source_test, y_pred_source, average='macro', zero_division=0)
print('[S-trained MLP on S_test]')
print(f'Accuracy: {source_accuracy:.6f}')
print(f'Macro Recall: {source_macro_recall:.6f}')
print('Classification Report:')
print(classification_report(y_source_test, y_pred_source, target_names=label_encoder.classes_, zero_division=0))


No-CORAL models already available in memory.
NO-CORAL CLASSIFIERS ON NO-CORAL SOURCE
[S-trained RF on S_test]
Accuracy: 0.999520
Macro Recall: 0.997836
Classification Report:
                  precision    recall  f1-score   support

          Benign       1.00      1.00      1.00    236142
            DDoS       1.00      1.00      1.00     25605
   DoS GoldenEye       1.00      1.00      1.00      2059
        DoS Hulk       1.00      1.00      1.00     46025
DoS Slowhttptest       1.00      0.99      0.99      1100
   DoS slowloris       0.99      1.00      0.99      1159
     FTP-Patator       1.00      1.00      1.00      1587
     SSH-Patator       1.00      1.00      1.00      1179

        accuracy                           1.00    314856
       macro avg       1.00      1.00      1.00    314856
    weighted avg       1.00      1.00      1.00    314856

[S-trained SVM on S_test]
Accuracy: 0.970190
Macro Recall: 0.756021
Classification Report:
                  precision    reca

In [20]:
### Evaluate no-CORAL classifiers on target ###

print('==============================================')
print('NO-CORAL CLASSIFIERS ON TARGET')
print('==============================================')

# Evaluate the no-coral RF on the target test split.
y_pred_target = rf_no_coral.predict(X_target_test)
target_accuracy = accuracy_score(y_target_test, y_pred_target)
target_macro_recall = recall_score(y_target_test, y_pred_target, average='macro', zero_division=0)
print('[S-trained RF on T_test]')
print(f'Accuracy: {target_accuracy:.6f}')
print(f'Macro Recall: {target_macro_recall:.6f}')
print('Classification Report:')
print(classification_report(y_target_test, y_pred_target, target_names=label_encoder.classes_, zero_division=0))

# Evaluate the no-coral SVM on the target test split.
y_pred_target = svm_no_coral.predict(X_target_test)
target_accuracy = accuracy_score(y_target_test, y_pred_target)
target_macro_recall = recall_score(y_target_test, y_pred_target, average='macro', zero_division=0)
print('[S-trained SVM on T_test]')
print(f'Accuracy: {target_accuracy:.6f}')
print(f'Macro Recall: {target_macro_recall:.6f}')
print('Classification Report:')
print(classification_report(y_target_test, y_pred_target, target_names=label_encoder.classes_, zero_division=0))

# Evaluate the no-coral MLP on the target test split.
y_pred_target = mlp_no_coral.predict(X_target_test)
target_accuracy = accuracy_score(y_target_test, y_pred_target)
target_macro_recall = recall_score(y_target_test, y_pred_target, average='macro', zero_division=0)
print('[S-trained MLP on T_test]')
print(f'Accuracy: {target_accuracy:.6f}')
print(f'Macro Recall: {target_macro_recall:.6f}')
print('Classification Report:')
print(classification_report(y_target_test, y_pred_target, target_names=label_encoder.classes_, zero_division=0))


NO-CORAL CLASSIFIERS ON TARGET
[S-trained RF on T_test]
Accuracy: 0.767600
Macro Recall: 0.240885
Classification Report:
                  precision    recall  f1-score   support

          Benign       0.76      1.00      0.87     27179
            DDoS       0.00      0.00      0.00      2000
   DoS GoldenEye       1.00      0.18      0.30       800
        DoS Hulk       0.00      0.00      0.00      2000
DoS Slowhttptest       0.00      0.00      0.00      1200
   DoS slowloris       1.00      0.75      0.86       600
     FTP-Patator       0.00      0.00      0.00      1200
     SSH-Patator       0.00      0.00      0.00      1200

        accuracy                           0.77     36179
       macro avg       0.35      0.24      0.25     36179
    weighted avg       0.61      0.77      0.67     36179

[S-trained SVM on T_test]
Accuracy: 0.755963
Macro Recall: 0.330941
Classification Report:
                  precision    recall  f1-score   support

          Benign       0.80   

In [21]:
### Evaluate with-CORAL classifiers on with-CORAL source ###

X_source_test_coral = X_source_test_coral_df
y_source_test_coral = source_test_df['Label'].astype(int)
X_target_test = target_test_df[shared_features]
y_target_test = target_test_df['Label'].astype(int)

print('==============================================')
print('WITH-CORAL CLASSIFIERS ON WITH-CORAL SOURCE')
print('==============================================')

# Evaluate the with-CORAL RF on the adapted source test split.
y_pred_source_coral = rf_coral.predict(X_source_test_coral)
source_accuracy_coral = accuracy_score(y_source_test_coral, y_pred_source_coral)
source_macro_recall_coral = recall_score(y_source_test_coral, y_pred_source_coral, average='macro', zero_division=0)
print('[S\'-trained RF on S\'_test]')
print(f'Accuracy: {source_accuracy_coral:.6f}')
print(f'Macro Recall: {source_macro_recall_coral:.6f}')
print('Classification Report:')
print(classification_report(y_source_test_coral, y_pred_source_coral, target_names=label_encoder.classes_, zero_division=0))

# Evaluate the with-CORAL SVM on the adapted source test split.
y_pred_source_coral = svm_coral.predict(X_source_test_coral)
source_accuracy_coral = accuracy_score(y_source_test_coral, y_pred_source_coral)
source_macro_recall_coral = recall_score(y_source_test_coral, y_pred_source_coral, average='macro', zero_division=0)
print('[S\'-trained SVM on S\'_test]')
print(f'Accuracy: {source_accuracy_coral:.6f}')
print(f'Macro Recall: {source_macro_recall_coral:.6f}')
print('Classification Report:')
print(classification_report(y_source_test_coral, y_pred_source_coral, target_names=label_encoder.classes_, zero_division=0))

# Evaluate the with-CORAL MLP on the adapted source test split.
y_pred_source_coral = mlp_coral.predict(X_source_test_coral)
source_accuracy_coral = accuracy_score(y_source_test_coral, y_pred_source_coral)
source_macro_recall_coral = recall_score(y_source_test_coral, y_pred_source_coral, average='macro', zero_division=0)
print('[S\'-trained MLP on S\'_test]')
print(f'Accuracy: {source_accuracy_coral:.6f}')
print(f'Macro Recall: {source_macro_recall_coral:.6f}')
print('Classification Report:')
print(classification_report(y_source_test_coral, y_pred_source_coral, target_names=label_encoder.classes_, zero_division=0))


WITH-CORAL CLASSIFIERS ON WITH-CORAL SOURCE
[S'-trained RF on S'_test]
Accuracy: 0.999514
Macro Recall: 0.995290
Classification Report:
                  precision    recall  f1-score   support

          Benign       1.00      1.00      1.00    236142
            DDoS       1.00      1.00      1.00     25605
   DoS GoldenEye       0.99      0.99      0.99      2059
        DoS Hulk       1.00      1.00      1.00     46025
DoS Slowhttptest       0.99      0.99      0.99      1100
   DoS slowloris       0.99      1.00      0.99      1159
     FTP-Patator       1.00      1.00      1.00      1587
     SSH-Patator       0.99      0.99      0.99      1179

        accuracy                           1.00    314856
       macro avg       1.00      1.00      1.00    314856
    weighted avg       1.00      1.00      1.00    314856

[S'-trained SVM on S'_test]
Accuracy: 0.952785
Macro Recall: 0.592345
Classification Report:
                  precision    recall  f1-score   support

          Ben

In [22]:
### Evaluate with-CORAL classifiers on target ###

print('==============================================')
print('WITH-CORAL CLASSIFIERS ON TARGET')
print('==============================================')

# Evaluate the with-CORAL RF on the target test split.
y_pred_target_coral = rf_coral.predict(X_target_test)
target_accuracy_coral = accuracy_score(y_target_test, y_pred_target_coral)
target_macro_recall_coral = recall_score(y_target_test, y_pred_target_coral, average='macro', zero_division=0)
print('[S\'-trained RF on T_test]')
print(f'Accuracy: {target_accuracy_coral:.6f}')
print(f'Macro Recall: {target_macro_recall_coral:.6f}')
print('Classification Report:')
print(classification_report(y_target_test, y_pred_target_coral, target_names=label_encoder.classes_, zero_division=0))

# Evaluate the with-CORAL SVM on the target test split.
y_pred_target_coral = svm_coral.predict(X_target_test)
target_accuracy_coral = accuracy_score(y_target_test, y_pred_target_coral)
target_macro_recall_coral = recall_score(y_target_test, y_pred_target_coral, average='macro', zero_division=0)
print('[S\'-trained SVM on T_test]')
print(f'Accuracy: {target_accuracy_coral:.6f}')
print(f'Macro Recall: {target_macro_recall_coral:.6f}')
print('Classification Report:')
print(classification_report(y_target_test, y_pred_target_coral, target_names=label_encoder.classes_, zero_division=0))

# Evaluate the with-CORAL MLP on the target test split.
y_pred_target_coral = mlp_coral.predict(X_target_test)
target_accuracy_coral = accuracy_score(y_target_test, y_pred_target_coral)
target_macro_recall_coral = recall_score(y_target_test, y_pred_target_coral, average='macro', zero_division=0)
print('[S\'-trained MLP on T_test]')
print(f'Accuracy: {target_accuracy_coral:.6f}')
print(f'Macro Recall: {target_macro_recall_coral:.6f}')
print('Classification Report:')
print(classification_report(y_target_test, y_pred_target_coral, target_names=label_encoder.classes_, zero_division=0))


WITH-CORAL CLASSIFIERS ON TARGET
[S'-trained RF on T_test]
Accuracy: 0.751237
Macro Recall: 0.125000
Classification Report:
                  precision    recall  f1-score   support

          Benign       0.75      1.00      0.86     27179
            DDoS       0.00      0.00      0.00      2000
   DoS GoldenEye       0.00      0.00      0.00       800
        DoS Hulk       0.00      0.00      0.00      2000
DoS Slowhttptest       0.00      0.00      0.00      1200
   DoS slowloris       0.00      0.00      0.00       600
     FTP-Patator       0.00      0.00      0.00      1200
     SSH-Patator       0.00      0.00      0.00      1200

        accuracy                           0.75     36179
       macro avg       0.09      0.12      0.11     36179
    weighted avg       0.56      0.75      0.64     36179

[S'-trained SVM on T_test]
Accuracy: 0.716604
Macro Recall: 0.257848
Classification Report:
                  precision    recall  f1-score   support

          Benign       0.9